[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ODJ0930/basalt/blob/main/Basalt_Qwen38_Uncensored_Colab.ipynb)

# Qwen3.8-Flash-Next abliterated (uncensored) on a Colab G4 (Basalt)

Runs [huihui-ai/Huihui-Qwen3.8-Flash-Next-abliterated-v2](https://huggingface.co/huihui-ai/Huihui-Qwen3.8-Flash-Next-abliterated-v2-GGUF)
(UD-IQ4_XS) with [Basalt](https://github.com/jesdga95/basalt) and serves an OpenAI / Anthropic compatible API.

* **Runtime:** G4 (RTX PRO 6000 Blackwell, 96 GB). Basalt needs compute capability 12.0.
* Basalt reads `.basalt` packs, so the GGUF is converted here. The drafter (MTP) and vision sections are copied from the official
  UD-IQ4_XS pack; the abliteration only changes the shared projections, the routed experts are the same.
* Disk plan: GGUF (94 GB) in a RAM disk, the official pack (98 GB) and the converted pack (98 GB) on the VM disk. After the
  conversion the GGUF and the official pack are deleted; only the uncensored pack stays.
* Measured on a G4: every expert in VRAM (55.4 GiB), ~220-280 tok/s on typical replies (a short code reply ran at ~370 tok/s).
  Context 262,144 tokens (the model's native length) by default.
* Time on a fresh VM: build ~2 min, downloads ~15-20 min, conversion ~5 min, loading ~3 min. With the pack saved in Google
  Drive (step 1), later runs copy it back instead of downloading and converting.
* **Recommended:** add a Hugging Face token as a Colab secret named `HF_TOKEN` (🔑 on the left, notebook access on).
  Hugging Face rate-limits large downloads (HTTP 429) even with a token; the downloads run one at a time and resume after a 429
  (the official pack with aria2, 8 connections), so a 429 only costs a pause.

### 1. Check the runtime, the token and Google Drive

With `USE_DRIVE` on, the converted pack is kept in Google Drive: the first run saves it there after the conversion, later runs
copy it back (~98 GB) and skip both downloads and the conversion. Mounting asks once for access to Drive: pick the account
that has the space. The API key is kept the same across runs: a Colab secret `BASALT_API_KEY` if you set one, else the key
saved in Drive next to the pack (made on the first run). For a fixed public URL, add the Colab secrets `CF_TUNNEL_TOKEN`
(a Cloudflare named tunnel's token) and `CF_PUBLIC_URL` (its hostname, whose service is `http://localhost:8090`); without
them step 6 opens a quick tunnel whose URL changes on every run.

In [ ]:
import os, subprocess, time
from pathlib import Path
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(gpu)
assert ", 12.0" in gpu, "Basalt needs a Blackwell card (compute capability 12.0): pick the G4 runtime"
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    Path("/content/hf_token").write_text(token)
    print("HF token: found in Colab secrets")
except Exception:
    print("HF token: none (anonymous download, may be rate-limited)")
USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_PACK = "/content/drive/MyDrive/basalt/Qwen3.8-FN-abliterated-v2-UD-IQ4_XS.basalt"  # @param {type:"string"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive copy:", "found: the downloads and the conversion are skipped" if Path(DRIVE_PACK).exists()
          else "none yet: the pack is saved there after the conversion")

# One API key across redeploys, so clients keep working: a Colab secret BASALT_API_KEY wins, then the key kept in Drive;
# otherwise a new key is made and (with Drive) kept there for next time.
import secrets
KEY = Path("/content/basalt_api_key.txt")
DRIVE_KEY = Path(DRIVE_PACK).parent / "basalt_api_key.txt"
try:
    KEY.write_text(userdata.get("BASALT_API_KEY").strip() + "\n")
    print("API key: from the Colab secret BASALT_API_KEY")
except Exception:
    if USE_DRIVE and DRIVE_KEY.exists():
        KEY.write_text(DRIVE_KEY.read_text())
        print("API key: the one kept in Drive (same as last time)")
    elif not KEY.exists():
        KEY.write_text(secrets.token_urlsafe(24) + "\n")
        print("API key: new")
    if USE_DRIVE and not DRIVE_KEY.exists():
        DRIVE_KEY.parent.mkdir(parents=True, exist_ok=True)
        DRIVE_KEY.write_text(KEY.read_text())
        print("API key: kept in Drive for next time")
KEY.chmod(0o600)

# Fixed public URL (optional): a Cloudflare named tunnel's token and hostname as Colab secrets; without them step 6 opens a
# quick tunnel whose URL changes on every run.
for secret, path in (("CF_TUNNEL_TOKEN", "/content/cf_tunnel_token"), ("CF_PUBLIC_URL", "/content/cf_public_url")):
    try:
        Path(path).write_text(userdata.get(secret).strip())
        Path(path).chmod(0o600)
    except Exception:
        pass
print("public URL:", "fixed, " + Path("/content/cf_public_url").read_text() if Path("/content/cf_tunnel_token").exists()
      else "quick tunnel (changes every run)")
!free -g | head -2; df -h / | tail -1

### 2. Build Basalt and download (background)

The engine build (~2 min) runs alongside the downloads. The GGUF (94 GB, into RAM) and then the official pack (98 GB, onto the disk) download one after the other, which avoids Hugging Face rate limits.

In [ ]:
!git clone -q --depth 1 https://github.com/ODJ0930/basalt.git /content/basalt 2>/dev/null || git -C /content/basalt pull -q
!mkdir -p /mnt/ramdisk && (mountpoint -q /mnt/ramdisk || mount -t tmpfs -o size=100G tmpfs /mnt/ramdisk)

GGUF_REPO = "huihui-ai/Huihui-Qwen3.8-Flash-Next-abliterated-v2-GGUF"
GGUF_DIR = "unsloth-UD-IQ4_XS"
BASE_PACK = "/content/models/Qwen3.8-FN-UD-IQ4_XS.basalt"
OUT_PACK = "/content/models/Qwen3.8-FN-abliterated-v2-UD-IQ4_XS.basalt"

def background(log, *cmd, **env):
    return subprocess.Popen(list(cmd), stdout=open(log, "w"), stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                            start_new_session=True, env={**os.environ, **env})

def done(log, mark):
    return Path(log).exists() and mark in Path(log).read_text(errors="replace")

jobs = {"build": (background("/content/build.log", "bash", "/content/basalt/colab/build.sh", MODELS="qwen4exp"), "BUILD_DONE")}
if not Path(OUT_PACK).exists() and USE_DRIVE and Path(DRIVE_PACK).exists():
    !mkdir -p /content/models
    jobs["drive"] = (background("/content/drive_copy.log", "bash", "-c",
        f"cp '{DRIVE_PACK}' '{OUT_PACK}.partial' && mv '{OUT_PACK}.partial' '{OUT_PACK}' && echo COPY_DONE"), "COPY_DONE")
elif not Path(OUT_PACK).exists():
    # one download at a time: two parallel ~100 GB Xet downloads hit Hugging Face's rate limit (HTTP 429)
    DL = "/content/basalt/colab/download.sh"
    jobs["download"] = (background("/content/download.log", "bash", "-c",
        f"MNT=/mnt/ramdisk REPO={GGUF_REPO} INCLUDE='{GGUF_DIR}/*' bash {DL} > /content/gguf.log 2>&1 && "
        f"MNT=/content/models REPO=jesdga/Qwen3.8-Flash-Next-Basalt PACK={Path(BASE_PACK).name} bash {DL} > /content/base.log 2>&1 && "
        "echo ALL_DOWNLOADS_DONE"), "ALL_DOWNLOADS_DONE")
logs = {"build": "/content/build.log", "download": "/content/download.log", "drive": "/content/drive_copy.log"}

start = time.time()
while not all(done(logs[k], mark) for k, (_, mark) in jobs.items()):
    for k, (proc, mark) in jobs.items():
        if proc.poll() not in (None, 0) and not done(logs[k], mark):
            !tail -20 {logs[k]}
            raise RuntimeError(f"{k} failed (see {logs[k]})")
    used = lambda p: subprocess.run(["du", "-sh", p], capture_output=True, text=True).stdout.split()[0] if Path(p).exists() else "0"
    got = (f"from Drive {used('/content/models')} / 98G" if "drive" in jobs else
           f"gguf {used('/mnt/ramdisk/' + GGUF_DIR)} / 94G, base {used('/content/models')} / 98G")
    print(f"[{(time.time() - start) / 60:4.1f} min] {got}, build {'done' if done(logs['build'], 'BUILD_DONE') else 'running'}", flush=True)
    time.sleep(60)
print("ready to convert" if "download" in jobs else "pack ready")

### 3. Convert the GGUF to a `.basalt` pack, then delete the GGUF and the official pack (skipped when the pack came from Drive)

`--compat-bf16`: the patched GGUF stores the hyper-connection matrices as Q8_0; Basalt reads them as BF16.

In [ ]:
if not Path(OUT_PACK).exists():
    first = sorted(Path("/mnt/ramdisk", GGUF_DIR).glob("*-00001-of-*.gguf"))[0]
    conv = background("/content/convert.log", "python3", "/content/basalt/tools/convert/convert.py",
                      "--gguf", str(first), "--sections-from", BASE_PACK, "--out", OUT_PACK,
                      "--name", "Qwen3.8-Flash-Next abliterated-v2 UD-IQ4_XS", "--compat-bf16")
    part = Path(OUT_PACK + ".partial")
    start = time.time()
    while conv.poll() is None:
        size = part.stat().st_size / 1e9 if part.exists() else 0
        print(f"[{(time.time() - start) / 60:4.1f} min] written {size:.1f} / ~98 GB", flush=True)
        time.sleep(60)
    !tail -n 5 /content/convert.log
    assert conv.returncode == 0 and Path(OUT_PACK).exists(), "conversion failed (see /content/convert.log)"

# keep only the uncensored pack
!rm -f {BASE_PACK}; rm -rf /mnt/ramdisk/{GGUF_DIR} /mnt/ramdisk/.hf /content/models/.cache /content/models/.hf
!umount /mnt/ramdisk 2>/dev/null; ls -l /content/models; df -h / | tail -1; free -g | head -2

# first run with Drive: save the pack there in the background (the next cells don't wait for it)
if USE_DRIVE and not Path(DRIVE_PACK).exists():
    Path(DRIVE_PACK).parent.mkdir(parents=True, exist_ok=True)
    background("/content/drive_save.log", "bash", "-c",
               f"cp '{OUT_PACK}' '{DRIVE_PACK}.partial' && mv '{DRIVE_PACK}.partial' '{DRIVE_PACK}' && echo SAVE_DONE")
    print("saving the pack to Drive in the background (log: /content/drive_save.log). The copy into Colab's Drive cache takes "
          "~5 min, the upload to Drive ~20 min more (~80 MB/s): keep the runtime until then, and expect the page to lag meanwhile")

### 4. Start the server and wait for the model to load

`CONTEXT` 262,144 is the model's native length; above 32,768 the KV cache grows only when a conversation needs it ("elastic KV").
`KV` empty keeps fp16; `int8` halves the KV memory. `VISION` `dynamic` accepts images (sent as `data:` URLs or uploads in the
panel) at no VRAM cost at rest; `off` refuses them. To change either later, edit and re-run this cell: `serve.sh` stops the
running server and loads again (~3 min), and the tunnel URL from step 6 keeps working.

In [ ]:
import json, urllib.request
CONTEXT = 262144  # @param {type:"integer"}
KV = ""  # @param ["", "int8", "q4_0", "k8v4"]
VISION = "dynamic"  # @param ["dynamic", "off"]
!PACK={OUT_PACK} CONTEXT={CONTEXT} KV={KV} VISION={VISION} bash /content/basalt/colab/serve.sh

def health():
    try:
        with urllib.request.urlopen("http://127.0.0.1:8090/health", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

for i in range(240):
    h = health()
    if h and h.get("loaded"):
        print("model loaded:", h.get("model"), "context", h.get("max_context"))
        break
    if subprocess.run(["pgrep", "-f", "[s]erve.server"], capture_output=True).returncode != 0:
        !tail -30 /content/basalt-server.log
        raise RuntimeError("the server stopped")
    if i % 3 == 0:
        print(f"[{i / 6:4.1f} min] loading {h.get('percent') if h else '...'}%", flush=True)
    time.sleep(10)
else:
    raise RuntimeError("the model did not load in 40 minutes")
!grep -hE "expert cache|every expert|elastic KV" ~/.cache/basalt/*/engine.log /content/basalt-server.log | tail -4

### 5. Quick test

In [ ]:
!python3 /content/basalt/colab/test.py

### 6. Public URL and web panel (optional)

Opens a Cloudflare quick tunnel. The API key is required; anyone with the URL **and** the key can use this runtime.
Show the key with `!cat /content/basalt_api_key.txt`.

In [ ]:
!bash /content/basalt/colab/tunnel.sh

### 7. Keep the runtime alive

Leave this cell running while you use the API. A running cell keeps Colab from reclaiming the runtime as idle (Colab Pro+ keeps
it in the background for up to ~24 h even with the tab closed). Every minute it also restarts the server if it exited and reopens
the tunnel if it stopped answering; a reopened quick tunnel has a **new URL**, printed here (same API key).

In [ ]:
!python3 /content/basalt/colab/keepalive.py